# ML Hackathon 2026: will this workout be hard for this person?

**Core idea.** The target `intensity` is a split into **tertiles, per (user_id, sport)**, of a sensor
measurement (most likely heart rate), computed on train + test combined.
Check: in the 193 groups, no train class exceeds N/3 by more than 1.
So the question is not "is this workout hard in absolute terms?" but
"is it harder than **this person's other workouts in this sport**?".

The pipeline in 6 steps:
1. features **relative** to the group (rank, z-score), without labels;
2. **time-neighbour** features: mean level of the known labels of the same group within ±0/1/3/12 months;
3. **nested OOF** so that these features never leak the label;
4. comparison of 3 model families in 5-fold CV;
5. post-processing: **duplicates** (label copying) then **tertile quotas** per group;
6. final training and `submission.csv`.

## How to read this notebook

Versions: v2 = base pipeline (16:09), v3 = + near-duplicates and exact quotas (17:10), v4 = + neighbourhood, bagging, ordinal assignment (17:50).

Each function is preceded by a cell that explains **which question it answers**, its inputs and its outputs.
The experiment cells (settings tried and rejected) are in `LOG.md` and summarised below: only the ideas that
gain ≥ +0.005 on average over 3 cross-validation seeds are kept.

**Results.** This notebook prints the 5-fold CV on seed 0 on this machine: `hist_gb` 0.614 after post-processing (versus 0.605 for the v2 code on the same seed). Decisions were made on seeds 1-3 (see `LOG.md`): v2 0.607 → v3 0.609 → v4 **0.619**. Kaggle public: v2 0.622, v3 0.623, v4: see `LOG.md`. The `submission.csv` written at the bottom is identical to the submitted file `submission_v4.csv`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

SEED = 0
CLASSES = ["easy", "moderate", "hard"]

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
y_train = train["intensity"].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
print(train.shape, test.shape)

(22728, 18) (7669, 17)


## Step 1: a train + test "pool" and the relative features

We stack train and test into a single table `pool`: rows `0 .. len(train)-1` are the train set,
the following ones the test set. `Y` holds the label (0 = easy, 1 = moderate, 2 = hard) for train and -1 for test.

**Why this is allowed:** these features use **no labels**, only the measurements.
Computing them on train + test is therefore leakage-free, and it is the same set on which
the tertiles were cut.

For each measure, within each (user, sport) group:
- `_rk`: percentile rank (0 = the lowest of the person's workouts, 1 = the highest). Robust to small groups (min. 9 workouts).
- `_z`: deviation from the group mean, in standard deviations.

We add a few derived measures (elevation gain per km, max speed / mean speed, ...) and
`speed_local`: the speed compared with the workouts of the same group within ±3 months (fitness changes over time).

In [2]:
pool = pd.concat([train.drop(columns="intensity"), test], ignore_index=True)
N_TRAIN = len(train)
TRAIN_IDX = np.arange(N_TRAIN)
TEST_IDX = np.arange(N_TRAIN, len(pool))
Y = np.full(len(pool), -1)
Y[TRAIN_IDX] = y_train

RAW = [c for c in test.columns if c != "workout_id"]   # colonnes d'origine, pour détecter les doublons

pool["t"] = (pool["year"] - pool["year"].min()) * 12 + pool["month"] - 1   # index du mois
pool["gid"] = pool.groupby(["user_id", "sport"]).ngroup()                  # identifiant du groupe
pool["group_size"] = pool.groupby("gid")["gid"].transform("size")

pool["gain_per_km"] = pool["altitude_gain_m"] / pool["distance_km"].clip(lower=0.1)
pool["climb_rate"] = pool["altitude_gain_m"] / pool["duration_min"].clip(lower=1)
pool["speed_ratio"] = pool["max_speed_kmh"] / pool["speed_kmh"].clip(lower=0.1)
pool["speed_cv"] = pool["speed_std_kmh"] / pool["speed_kmh"].clip(lower=0.1)

REL = ["duration_min", "distance_km", "speed_kmh", "max_speed_kmh", "speed_std_kmh",
       "altitude_mean_m", "altitude_gain_m", "altitude_loss_m", "n_points", "start_hour_utc",
       "gain_per_km", "climb_rate", "speed_ratio", "speed_cv"]
grp = pool.groupby("gid")
for c in REL:
    pool[c + "_rk"] = grp[c].rank(pct=True)
    pool[c + "_z"] = (pool[c] - grp[c].transform("mean")) / (grp[c].transform("std") + 1e-9)

# One-hot pour les variables catégorielles (nécessaire pour la régression logistique)
pool = pd.concat([pool, pd.get_dummies(pool[["sport", "gender"]], dtype=int)], axis=1)
ONEHOT = [c for c in pool.columns if c.startswith(("sport_", "gender_"))]

## Step 2: rolling sums over time

A single function serves all the "time-neighbour" features.
We arrange the values in a `group × month` grid, take a cumulative sum along the months,
and the sum over a window `[t - w, t + w]` becomes a simple difference of two grid cells.

`window_stats(fit, apply, values, w)` answers, for each row of `apply`:
"among the rows of `fit` from the **same group** and **at most w months** away, what is the sum of the values and how many are there?"

### Function `window_stats(fit_idx, apply_idx, values, w)`

**Question it answers:** for each row of `apply_idx`, "among the rows of `fit_idx` from the same (user, sport) group
and at most `w` months away, what is the sum of `values` and how many are there?"

**How:** we fill a grid `S[group, month]` with the sums and `C[group, month]` with the counts (`np.add.at` accumulates
even when several rows fall into the same cell). A cumulative sum along the months then turns the sum over a
window `[t-w, t+w]` into a simple difference `S[g, t+w+1] - S[g, t-w]`. This is O(n) instead of O(n²): the same function serves
`speed_local` (without labels) and the time neighbours (with labels).

**Inputs:** `fit_idx` = rows whose value we are allowed to use; `apply_idx` = rows for which the feature is computed;
`values` = what is summed (speed or label 0/1/2); `w` = half-width of the window in months. **Outputs:** two vectors (sum, count).

In [3]:
N_G = pool["gid"].max() + 1
N_T = pool["t"].max() + 1
GID = pool["gid"].to_numpy()
T = pool["t"].to_numpy()


def window_stats(fit_idx, apply_idx, values, w):
    S = np.zeros((N_G, N_T + 1))
    C = np.zeros((N_G, N_T + 1))
    np.add.at(S, (GID[fit_idx], T[fit_idx] + 1), values)
    np.add.at(C, (GID[fit_idx], T[fit_idx] + 1), 1)
    S, C = S.cumsum(1), C.cumsum(1)          # S[g, k] = somme des mois < k
    g, t = GID[apply_idx], T[apply_idx]
    hi = np.minimum(t + w, N_T - 1) + 1
    lo = np.maximum(t - w, 0)
    return S[g, hi] - S[g, lo], C[g, hi] - C[g, lo]


# Vitesse locale (sans étiquette) : vitesse / moyenne des AUTRES séances du groupe à ±3 mois
s, n = window_stats(np.arange(len(pool)), np.arange(len(pool)), pool["speed_kmh"].to_numpy(), 3)
pool["speed_local"] = (pool["speed_kmh"] / ((s - pool["speed_kmh"]) / (n - 1)).replace(0, np.nan)).fillna(1.0)

BASE = (["year", "month", "weekday", "start_hour_utc", "group_size", "speed_local"]
        + REL + [c + "_rk" for c in REL] + [c + "_z" for c in REL] + ONEHOT)
BASE = list(dict.fromkeys(BASE))   # "start_hour_utc" was listed twice
print(len(BASE), "features de base")

57

 features de base


## Step 3: "neighbour" features built from the labels, without leakage

`label_features(fit, apply)`: for each row of `apply`, the mean of the labels (0/1/2) of the `fit` workouts
from the same group within ±0, 1, 3, 12 months, plus their count.
- The mean is **smoothed towards 1** (moderate) with a weight `PRIOR = 3`: with only 1 neighbour, we do not trust it completely.
- The **number** of neighbours is also a feature: the model knows whether the mean is reliable.

**The anti-leakage rule:** a row never sees its own label.
- Rows to predict (validation, test): `fit` = all the training rows, disjoint by construction.
- Training rows: `oof_label_features` splits the training set into 5 parts; each part
  receives its features computed from the 4 others (OOF = *out-of-fold*).

In the CV, all of this is redone **inside each fold** with only the labels of the training fold
(nested OOF). Otherwise the validation labels would enter the training features and the validation score would be too optimistic.

### Functions `time_label_features`, `oof_label_features`, `make_X`

* **`time_label_features(fit_idx, apply_idx)`**: for each row to predict, the mean of the known labels (0 = easy, 1 = moderate, 2 = hard)
  of the workouts of the same group within ±0, ±1, ±3 and ±12 months, plus their count. The mean is *smoothed* towards 1 (moderate) with a weight
  `PRIOR = 3`: `(somme + 3·1) / (n + 3)`. With a single neighbour, the feature stays close to 1; with 20 neighbours, it follows the neighbours.
  The count `nbW_n` tells the model whether the mean is reliable.
* **`oof_label_features(idx)`**: the same thing for the **training** rows, but *out-of-fold*: we split `idx` into 5 parts and
  each part receives its features computed from the 4 others. Without this, a training row would see its own label in
  the feature, the model would learn "the feature = the answer" and would collapse on the test set, where the label is not in the feature.
* **`make_X(idx, nb)`**: assembles the final matrix = base columns (`BASE`) + label features (`nb`).

**Two known limitations (17:40 audit), which do not affect the honesty of the validation:**
1. Training rows see 4/5 of the known labels (internal OOF), rows to predict see 100%: the counters `nb*_n` and `kern_w` are ~25% larger at prediction time than at training time. The CV reproduces exactly the same shift, so the validation score remains honest; the smoothed means, for their part, move by less than 1%.
2. 505 pairs of identical workouts exist in the train set (503 with the same label). The internal `KFold` does not keep both twins in the same fold, so for ~4.7% of the training rows the twin's label enters their own features (`kern_mean`, `nb0_mean`). This is not leakage into the validation or the test set (those rows are treated identically), it is a slight train/apply shift. Fix tested in v5: `GroupKFold` by duplicate cluster (see `LOG.md`).

In [4]:
WINDOWS = [0, 1, 3, 12]
PRIOR = 3.0


def time_label_features(fit_idx, apply_idx):
    out = {}
    for w in WINDOWS:
        s, n = window_stats(fit_idx, apply_idx, Y[fit_idx].astype(float), w)
        out[f"nb{w}_mean"] = (s + PRIOR * 1.0) / (n + PRIOR)
        out[f"nb{w}_n"] = n
    return pd.DataFrame(out)


def oof_label_features(idx, k=5):
    parts = []
    for a, b in KFold(k, shuffle=True, random_state=SEED).split(idx):
        f = label_features(idx[a], idx[b])
        f.index = b
        parts.append(f)
    return pd.concat(parts).sort_index()


def make_X(idx, nb):
    return pd.concat([pool.loc[idx, BASE].reset_index(drop=True), nb.reset_index(drop=True)], axis=1)

## Step 3b: two new features built from the labels (added after the 2nd round of experiments)

**a) Neighbours at a similar speed (Gaussian kernel).** For each workout, the mean of the known labels of the same
(user, sport), weighted by `w = exp(-Δrang_vitesse² / 0.01)`: a workout at a very similar speed counts almost 1,
a workout 0.2 away in rank hardly counts any more. We add the sum of the weights (reliability of the mean).
Idea: "the other times this person rode/ran at this speed, was it hard or easy?"

**b) Speed→difficulty relationship specific to each person.** Within each group, Spearman correlation
(correlation between ranks) between the label and the speed rank, and between the label and the elevation-gain rank.
For some people, going fast = hard; for others, it is the climbing that matters. We also add the product
correlation × rank of the workout.

Both use the labels, so they go through the same anti-leakage rule as the temporal neighbours:
`label_features` is called in nested OOF, and a row never sees its own label.
Validation on CV seeds 1, 2, 3: 0.6067 ± 0.0005 versus 0.6017 ± 0.0005 for the baseline (+0.005 on each seed).

### Functions `speed_kernel_features`, `group_correlation_features`, `label_features`

* **`speed_kernel_features(fit_idx, apply_idx)`**: the "similar speed" variant of the neighbours. For each workout, mean of the known labels
  of the same group, weighted by `w = exp(-Δrang_vitesse² / 0.01)`: a workout at the same speed rank counts 1, a workout whose rank differs by
  0.1 counts `e^-1 ≈ 0.37`, by 0.2 almost 0. Answers the question: "the other times this person went at this speed, was it hard?".
  Returns the smoothed mean (`prior = 1`) and the sum of the weights (reliability).
* **`group_correlation_features(fit_idx, apply_idx)`**: within each group, Spearman correlation (= Pearson correlation between the ranks)
  between the label and the speed rank (`rho_speed`), and between the label and the elevation-gain rank (`rho_gain`). For some people
  difficulty follows speed, for others elevation gain: the model can adapt the importance of each column to the person. We add
  the products `rho × rang de la séance`, which are directly a "personalised predicted difficulty".
* **`label_features(fit_idx, apply_idx)`**: concatenates the three families above. It is the only function called by the rest of the code,
  so the anti-leakage rule (a row never sees its own label) is enforced everywhere in one go.

In [5]:
SPEED_RK = pool["speed_kmh_rk"].to_numpy()
GAIN_RK = pool["altitude_gain_m_rk"].to_numpy()


def speed_kernel_features(fit_idx, apply_idx, inv_bw=1 / 0.01, prior=1.0):
    S = np.zeros(len(apply_idx))
    W = np.zeros(len(apply_idx))
    for gid in np.unique(GID[apply_idx]):
        F = fit_idx[GID[fit_idx] == gid]
        pos = np.where(GID[apply_idx] == gid)[0]
        if len(F) == 0:
            continue
        d = SPEED_RK[apply_idx[pos]][:, None] - SPEED_RK[F][None, :]
        w = np.exp(-d * d * inv_bw)
        S[pos] = w @ Y[F]
        W[pos] = w.sum(1)
    return (S + prior * 1.0) / (W + prior), W


def group_correlation_features(fit_idx, apply_idx):
    df = pd.DataFrame({"g": GID[fit_idx], "y": Y[fit_idx], "s": SPEED_RK[fit_idx], "a": GAIN_RK[fit_idx]})
    r = df.groupby("g")[["y", "s", "a"]].rank()          # Spearman = corrélation de Pearson entre rangs
    r["g"] = df["g"].to_numpy()
    with np.errstate(divide="ignore", invalid="ignore"):
        rho = r.groupby("g")[["y", "s", "a"]].apply(lambda d: pd.Series({"s": d["y"].corr(d["s"]), "a": d["y"].corr(d["a"])}))
    rho = rho.reindex(range(N_G)).fillna(0.0)
    ag = GID[apply_idx]
    rs, ra = rho["s"].to_numpy()[ag], rho["a"].to_numpy()[ag]
    return {"rho_speed": rs, "rho_gain": ra,
            "rho_speed_x_rk": rs * SPEED_RK[apply_idx], "rho_gain_x_rk": ra * GAIN_RK[apply_idx]}


def label_features(fit_idx, apply_idx):
    out = time_label_features(fit_idx, apply_idx)
    out["kern_mean"], out["kern_w"] = speed_kernel_features(fit_idx, apply_idx)
    for k, v in group_correlation_features(fit_idx, apply_idx).items():
        out[k] = v
    return out

## Step 3c (v4): three more families of neighbourhood features

Validated in CV on 5 seeds (+0.0035 on average, positive on every seed). All of them go through `label_features`,
hence through the same anti-leakage rule (nested OOF, a row never sees its own label).

* **`knn_features`**: for each workout, we look for the k = 5 and k = 15 *labelled* workouts of the same group that are
  closest in the z-score space (speed, duration, elevation gain, max speed, speed standard deviation) and take the
  mean of their labels (smoothed towards 1) and the mean distance (reliability). Answers the question: "this person's most
  similar workouts, were they hard?". More correlated with the label (Spearman 0.48) than the kernel on
  speed alone (0.42).
* **`residual_kernel_features`**: the *residual* `étiquette − 2 × rang de vitesse` of a known workout says whether it was
  harder than its speed suggested. We average it, weighted by a time kernel
  (`exp(−Δmois²/18)`) and by a speed kernel: "right now / at this pace, is this person above
  or below their usual speed→difficulty relationship?".
* **`trend_features`**: slope (ridge) of the label as a function of the month within the group, and mean of the labels over the
  3 previous months vs the 3 following months (asymmetric windows): fitness rises or falls over the months.

`label_features` is then redefined to concatenate the v2 features and these new features.

In [6]:
Z_COLS = [c + "_z" for c in ["speed_kmh", "duration_min", "altitude_gain_m", "max_speed_kmh", "speed_std_kmh"]]
ZM = pool[Z_COLS].to_numpy(dtype=float)
T_F = T.astype(float)


def knn_features(fit_idx, apply_idx, ks=(5, 15), prior=1.0):
    """mean label of the k nearest known rows of the same group in standardised feature space + mean distance."""
    out = {f"knn{k}_mean": np.full(len(apply_idx), 1.0) for k in ks}
    for k in ks:
        out[f"knn{k}_dist"] = np.full(len(apply_idx), 5.0)
    ag = GID[apply_idx]
    for gid in np.unique(ag):
        F = fit_idx[GID[fit_idx] == gid]; pos = np.where(ag == gid)[0]
        if len(F) == 0:
            continue
        A = ZM[apply_idx[pos]]; B = ZM[F]
        D = np.sqrt(np.maximum((A * A).sum(1)[:, None] + (B * B).sum(1)[None, :] - 2 * A @ B.T, 0))
        yF = Y[F].astype(float)
        for k in ks:
            kk = min(k, len(F))
            nn = np.argpartition(D, kk - 1, axis=1)[:, :kk] if kk < len(F) else np.tile(np.arange(len(F)), (len(pos), 1))
            out[f"knn{k}_mean"][pos] = (yF[nn].sum(1) + prior) / (kk + prior)
            out[f"knn{k}_dist"][pos] = np.take_along_axis(D, nn, 1).mean(1)
    return out


def residual_kernel_features(fit_idx, apply_idx, time_bw=18.0, speed_inv_bw=1 / 0.01, prior=1.0):
    """kernel-weighted mean of (label - 2*speed_rk) of known rows: is this period / this speed harder than speed alone says."""
    res_all = Y.astype(float) - 2.0 * SPEED_RK
    St = np.zeros(len(apply_idx)); Wt = np.zeros(len(apply_idx)); Ss = np.zeros(len(apply_idx)); Ws = np.zeros(len(apply_idx))
    ag = GID[apply_idx]
    for gid in np.unique(ag):
        F = fit_idx[GID[fit_idx] == gid]; pos = np.where(ag == gid)[0]
        if len(F) == 0:
            continue
        r = res_all[F]
        dt = T_F[apply_idx[pos]][:, None] - T_F[F][None, :]
        wt = np.exp(-dt * dt / time_bw)
        St[pos] = wt @ r; Wt[pos] = wt.sum(1)
        ds = SPEED_RK[apply_idx[pos]][:, None] - SPEED_RK[F][None, :]
        ws = np.exp(-ds * ds * speed_inv_bw)
        Ss[pos] = ws @ r; Ws[pos] = ws.sum(1)
    return {"res_time": St / (Wt + prior), "res_time_w": Wt, "res_speed": Ss / (Ws + prior)}


def trend_features(fit_idx, apply_idx, prior=PRIOR):
    """slope of known label vs month within group; label mean of previous 3 months vs next 3 months (asymmetric windows)."""
    df = pd.DataFrame({"g": GID[fit_idx], "t": T_F[fit_idx], "y": Y[fit_idx].astype(float)})
    gm = df.groupby("g").agg(mt=("t", "mean"), my=("y", "mean"), n=("t", "size"))
    df = df.join(gm, on="g")
    df["dt"] = df["t"] - df["mt"]; df["cov"] = df["dt"] * (df["y"] - df["my"]); df["var"] = df["dt"] ** 2
    agg = df.groupby("g")[["cov", "var"]].sum()
    slope = (agg["cov"] / (agg["var"] + 10.0)).reindex(range(N_G)).fillna(0.0).to_numpy()   # ridge-shrunk slope per month
    mt = gm["mt"].reindex(range(N_G)).fillna(0.0).to_numpy()
    ag = GID[apply_idx]
    out = {"trend_slope": slope[ag], "trend_slope_x_dt": slope[ag] * (T_F[apply_idx] - mt[ag])}
    S = np.zeros((N_G, N_T + 1)); C = np.zeros((N_G, N_T + 1))
    np.add.at(S, (GID[fit_idx], T[fit_idx] + 1), Y[fit_idx].astype(float)); np.add.at(C, (GID[fit_idx], T[fit_idx] + 1), 1)
    S, C = S.cumsum(1), C.cumsum(1)
    t = T[apply_idx]
    lo = np.maximum(t - 3, 0); s_prev = S[ag, t] - S[ag, lo]; n_prev = C[ag, t] - C[ag, lo]
    hi = np.minimum(t + 3, N_T - 1) + 1; s_next = S[ag, hi] - S[ag, t + 1]; n_next = C[ag, hi] - C[ag, t + 1]
    out["prev3_mean"] = (s_prev + prior) / (n_prev + prior); out["prev3_n"] = n_prev
    out["next3_mean"] = (s_next + prior) / (n_next + prior); out["next3_n"] = n_next
    out["next_minus_prev"] = out["next3_mean"] - out["prev3_mean"]
    return out


_base_lf_v4 = label_features


def label_features(fit_idx, apply_idx):
    out = _base_lf_v4(fit_idx, apply_idx)
    for fn in (knn_features, residual_kernel_features, trend_features):
        for k, v in fn(fit_idx, apply_idx).items():
            out[k] = v
    return out

## Step 4: post-processing

**Duplicates.** 505 groups of identical workouts in the train set (only 2 with different labels),
and 327 test rows identical to a train row: the same workout recorded twice.
`duplicate_labels` copies the known label (majority vote) onto the copy. This is competition data, not external data.

**Tertile quotas.** In a group of N workouts (train + test), each class has ≈ N/3 of them.
We already know the labels of `known` (and of the copied duplicates), so we know how many of each class
remain to be assigned among the rows to predict. `balance`:
1. computes this remaining quota per class (rounded so that it adds up exactly to the number of rows);
2. sorts the rows of the group by **predicted difficulty** = P(moderate) + 2·P(hard);
3. gives `easy` to the q₀ lowest, `moderate` to the next q₁, `hard` to the rest.

This is exactly the mechanism that created the labels: a tertile split of a ranking.

**Honest validation:** in CV, `apply` = validation + test, `known` = training fold. We never use
the actual counts of the validation set, just as for the real test set.

### Functions `duplicate_labels`, `integer_quota`, `balance`

* **`near_duplicate_labels(known_idx, apply_idx, fixed)`** (v3): after the exact duplicates, looks, for each remaining row, for a
  *known* workout in the same slot (user, sport, year, month, day of week, hour) whose duration, distance, speed and max speed are
  within ±1%: the same workout re-recorded with different rounding. In the train set, 91% of these pairs have the same label, so we copy it.
  Validated gain: +0.0017 macro-F1 on average over 3 seeds, positive on each of them. In total 327 exact copies + 71 near-duplicates = **398 copied labels** on the test set (the number printed at the bottom).
* **`target_counts(n)`** (v3): the exact quotas of a group of `n` workouts. When `n` is not a multiple of 3, the analysis of the train
  counts shows that the extra workout goes to `hard`, then to `moderate`: (k, k, k+1) and (k, k+1, k+1). `balance` uses it and only falls
  back on `integer_quota` if the known counts are inconsistent with the rule (never the case here).
* **`duplicate_labels(known_idx, apply_idx)`**: `ROW_KEY` is a hash of the 16 original columns of each row. Two rows with the same
  hash are the same workout recorded twice. For each row to predict, we look for a *known* row with the same hash and copy
  its label (majority vote if there are several). Returns −1 when there is no duplicate. 327 test rows are affected.
* **`integer_quota(rem, m)`**: turns real-valued quotas (e.g. 4.3 / 3.3 / 2.4) into integers that sum exactly to `m` (the number of
  rows to assign), rounding up the largest fractional parts.
* **`balance(known_idx, apply_idx, proba, fixed)`**: the key post-processing step. For each (user, sport) group of size N:
  1. we know that each class has ≈ N/3 workouts (checked: no class in the train set exceeds N/3 + 1);
  2. we subtract the labels already known (train + copied duplicates) → remaining quota per class;
  3. we sort the rows to predict by **predicted difficulty** `score = P(moderate) + 2·P(hard)`;
  4. the `q₀` lowest get `easy`, the next `q₁` get `moderate`, the rest `hard`.

  The model is therefore only used to **rank** a person's workouts against one another; the cut-off is imposed by the quotas, exactly
  the way the labels were created. This is what takes the HGB from 0.587 (argmax) to 0.605 in validation.

In [7]:
ROW_KEY = pd.util.hash_pandas_object(pool[RAW], index=False).to_numpy()


# Quasi-doublons stricts : même créneau (user, sport, année, mois, jour, heure) et les 4 mesures à ±1 %.
# Dans le train, 164 lignes ont un tel voisin (hors copie exacte) et son étiquette est la même dans 91 % des cas.
MEAS = pool[["duration_min", "distance_km", "speed_kmh", "max_speed_kmh"]].to_numpy()
SLOT_H = pool.groupby(["user_id", "sport", "year", "month", "weekday", "start_hour_utc"]).ngroup().to_numpy()


def near_duplicate_labels(known_idx, apply_idx, fixed, tol=0.01):
    out = fixed.copy()
    by_slot = pd.Series(known_idx).groupby(SLOT_H[known_idx]).apply(np.array).to_dict()
    for i in np.where(fixed < 0)[0]:                      # seulement les lignes sans doublon exact
        a = apply_idx[i]
        F = by_slot.get(SLOT_H[a])
        if F is None:
            continue
        rel = np.abs(MEAS[F] - MEAS[a]) / np.maximum(np.abs(MEAS[a]), 1e-6)
        m = (rel <= tol).all(1)
        if m.any():
            out[i] = np.bincount(Y[F][m], minlength=3).argmax()   # vote majoritaire
    return out


# Règle exacte des quotas : N = 3k + r séances dans le groupe -> (k, k, k+1) si r = 1, (k, k+1, k+1) si r = 2.
# Vérifiée sur les comptes du train : la séance « en trop » va toujours aux classes du haut.
def target_counts(n):
    k, r = divmod(n, 3)
    return np.array([k, k + (r >= 2), k + (r >= 1)])


def duplicate_labels(known_idx, apply_idx):
    lab = pd.Series(Y[known_idx], index=ROW_KEY[known_idx])
    majority = lab.groupby(level=0).agg(lambda s: s.mode().iloc[0])
    return pd.Series(ROW_KEY[apply_idx]).map(majority).fillna(-1).astype(int).to_numpy()


def integer_quota(rem, m):
    rem = np.clip(rem, 0, None)
    rem = np.ones(3) if rem.sum() == 0 else rem
    rem = rem * m / rem.sum()
    q = np.floor(rem).astype(int)
    for i in np.argsort(-(rem - q))[: m - q.sum()]:
        q[i] += 1
    return q


def balance(known_idx, apply_idx, proba, fixed):
    pred = fixed.copy()
    score = proba @ np.array([0.0, 1.0, 2.0])
    known_counts = np.zeros((N_G, 3))
    np.add.at(known_counts, (GID[known_idx], Y[known_idx]), 1)
    fx = fixed >= 0
    np.add.at(known_counts, (GID[apply_idx][fx], fixed[fx]), 1)
    group_n = pool.groupby("gid").size().to_numpy()
    apply_gid = GID[apply_idx]
    for g in np.unique(apply_gid):
        rows = np.where((apply_gid == g) & ~fx)[0]
        if len(rows) == 0:
            continue
        rem = target_counts(group_n[g]) - known_counts[g]
        q = rem.astype(int) if (rem >= 0).all() and rem.sum() == len(rows) else integer_quota(rem, len(rows))
        order = rows[np.argsort(score[rows], kind="stable")]
        pred[order] = np.repeat([0, 1, 2], q)
    return pred

## Step 4b (v4): assignment under quotas by ordinal cost, and bagging

**`balance_ord`** keeps exactly the quotas of `balance`, but replaces "sort by predicted difficulty, then cut"
with an **optimal assignment**: we create one "seat" per workout to assign, labelled easy / moderate / hard according to the
quotas, and assign each workout to a seat so as to minimise the sum of expected ordinal errors
`Σ_k P(k) · |k − seat class|` (`scipy.optimize.linear_sum_assignment`, 10-20 ms per group).
For easy vs hard this is equivalent to sorting; the difference comes from the *moderate* column: a workout that the model
truly judges moderate stays in the middle, instead of being pushed out by a workout with flat probabilities.
Validated: +0.0022 in CV, positive on 5 seeds out of 5. The −log p cost (maximum likelihood) does worse (−0.0008).

**Bagging**: the final model is the average of the probabilities of 5 `HistGradientBoostingClassifier` models, identical except for
`random_state` (which changes the early-stopping split), via a `VotingClassifier(voting="soft")`. Pure variance
reduction: +0.0026 in CV, positive on all 3 seeds.

In [8]:
from scipy.optimize import linear_sum_assignment
from sklearn.ensemble import VotingClassifier

GROUP_N = pool.groupby("gid").size().to_numpy()


def balance_ord(known_idx, apply_idx, proba, fixed):
    """Quota-constrained assignment minimising the expected ordinal error sum_k P[row,k]*|k-c|."""
    D = np.abs(np.subtract.outer(np.arange(3), np.arange(3))).astype(float)   # D[k, c] = |k - c|
    pred = fixed.copy()
    kc = np.zeros((N_G, 3)); np.add.at(kc, (GID[known_idx], Y[known_idx]), 1)
    fx = fixed >= 0; np.add.at(kc, (GID[apply_idx][fx], fixed[fx]), 1)
    ag = GID[apply_idx]
    for g in np.unique(ag):
        rows = np.where((ag == g) & ~fx)[0]
        if len(rows) == 0:
            continue
        rem = target_counts(GROUP_N[g]) - kc[g]
        q = rem.astype(int) if (rem >= 0).all() and rem.sum() == len(rows) else integer_quota(rem, len(rows))
        slots = np.repeat([0, 1, 2], q)
        cost = (proba[rows] @ D)[:, slots]
        r, c = linear_sum_assignment(cost)
        pred[rows[r]] = slots[c]
    return pred


def hgb(seed):
    return HistGradientBoostingClassifier(learning_rate=0.05, max_iter=400, max_leaf_nodes=31,
                                          l2_regularization=1.0, random_state=seed)


def bagged_hgb(n_seeds=5):
    return VotingClassifier([(f"hgb{s}", hgb(s)) for s in range(n_seeds)], voting="soft")

## Step 5: 5-fold cross-validation and 3 model families

For each outer fold:
1. neighbour features of the training rows via inner OOF (step 3);
2. neighbour features of validation + test from the training labels only;
3. model training, then probabilities on validation + test;
4. three macro-F1 scores on the validation set: **raw** (argmax), **+ duplicates**, **+ duplicates + quotas**.

The 3 families:
- **LogisticRegression** (linear, standardised variables): the simple baseline;
- **RandomForest**: average of many deep trees, captures non-linear effects;
- **HistGradientBoosting**: trees built one after another, each one correcting the errors of the previous ones. Detail: with more than 10,000 rows, scikit-learn enables automatic early stopping (10% of the train set held out, stopping after 10 iterations without log-loss improvement), so `max_iter = 400` is a ceiling, not the actual number of trees. This is also why changing `random_state` gives different models, which is what makes the v4 bagging useful.

### Functions `macro_f1` and `run_cv`

* **`macro_f1(y_true, y_pred)`**: the Kaggle metric. F1 of each class, then a simple average: each class counts for one third.
* **`run_cv(make_model)`**: stratified 5-fold cross-validation. In each fold: (1) label features of the training rows
  in inner OOF; (2) features of the validation rows **and of the test set** from the labels of the training fold only
  (the test set is included because the quotas are shared between validation and test within each group); (3) training and probabilities;
  (4) three scores on the validation set: `raw` = argmax, `dup` = + duplicates, `full` = + duplicates + quotas. Returns the scores and the
  out-of-fold predictions, which are used for the confusion matrix.

  Note: in CV, the validation rows share the quotas with the test rows, whereas on the real test set the quotas are
  exact. CV therefore underestimates the real score (0.607 in CV → 0.622 on the public leaderboard).

In [9]:
MODELS = {
    "logreg": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.5)),
    "random_forest": lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                                    max_features=0.3, n_jobs=-1, random_state=SEED),
    "hist_gb": lambda: HistGradientBoostingClassifier(learning_rate=0.05, max_iter=400, max_leaf_nodes=31,
                                                      l2_regularization=1.0, random_state=SEED),
}


def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro")


def run_cv(make_model, n_splits=5):
    preds = {k: np.zeros(N_TRAIN, int) for k in ["raw", "dup", "full"]}
    for fit, val in StratifiedKFold(n_splits, shuffle=True, random_state=SEED).split(TRAIN_IDX, y_train):
        X_fit = make_X(fit, oof_label_features(fit))
        apply = np.concatenate([val, TEST_IDX])
        X_apply = make_X(apply, label_features(fit, apply))
        model = make_model().fit(X_fit, Y[fit])
        proba = model.predict_proba(X_apply)
        fixed = near_duplicate_labels(fit, apply, duplicate_labels(fit, apply))
        raw = proba.argmax(1)
        preds["raw"][val] = raw[: len(val)]
        preds["dup"][val] = np.where(fixed >= 0, fixed, raw)[: len(val)]
        preds["full"][val] = balance_ord(fit, apply, proba, fixed)[: len(val)]
    scores = {k: round(macro_f1(y_train, p), 4) for k, p in preds.items()}
    return scores, preds


results = {}
for name, make_model in MODELS.items():
    results[name] = run_cv(make_model)
    print(f"{name:14s}", results[name][0])

C:\Users\linjo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\_function_base_impl.py:3057: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)


logreg         {'raw': 0.5763, 'dup': 0.5912, 'full': 0.6}


C:\Users\linjo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\_function_base_impl.py:3057: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)


random_forest  {'raw': 0.5969, 'dup': 0.6015, 'full': 0.6102}


C:\Users\linjo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\_function_base_impl.py:3057: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)


hist_gb        {'raw': 0.5933, 'dup': 0.6024, 'full': 0.614}


### Ideas tested and not kept

Same protocol (5-fold CV `random_state=0`, nested OOF, duplicates then quotas). Rule: keep only a gain ≥ +0.003.

| Idea | Best CV score | Decision |
| --- | --- | --- |
| HGB baseline (lr 0.05, 31 leaves, l2 1) | 0.6051 | |
| 1. Average of HGB + RF probabilities (0.5/0.5, 0.7/0.3, 0.3/0.7) | 0.6055 | dropped (+0.0004) |
| 2. HGB tuning (15 combinations): lr 0.03, 63 leaves, l2 0 | 0.6103 | gain that turned out to be noise (see 5) |
| 3a. easy / hard proportions among the neighbours | 0.6061 | dropped |
| 3b. Neighbour mean weighted by exp(-\|Δmonths\|/3) | 0.6060 | dropped |
| 4. Rank by P(hard) − P(easy) instead of P(moderate) + 2·P(hard) | 0.6103 | no effect |
| 5. Stability over seeds 1, 2, 3: baseline vs tuned HGB | 0.6017 ± 0.0005 vs 0.6016 ± 0.0014 | mean gain −0.0001 → tuning dropped |

Lesson: the best of the 15 settings was the best **on that particular split**. On three other splits, it does no better
than the baseline. Picking the maximum among many trials on the same CV gives an over-optimistic score.
We therefore keep the baseline model.

### Second round: new information (mean over CV seeds 1, 2, 3; threshold +0.005)

| Idea | Mean score (± std) | Gain | Decision |
| --- | --- | --- | --- |
| Baseline | 0.6017 ± 0.0005 | | |
| 1. Neighbours weighted by a Gaussian kernel on date + speed rank | 0.6035 ± 0.0021 | +0.0018 | no |
| 1a. Kernel on speed rank only | 0.6042 ± 0.0022 | +0.0026 | no (best variant) |
| 1b. Date + speed + duration rank | 0.6037 ± 0.0007 | +0.0021 | no |
| 1c. Date + speed + max speed rank | 0.6031 ± 0.0030 | +0.0014 | no |
| 2. Spearman correlation label ~ speed rank / elevation gain rank, per group (OOF) | 0.6056 ± 0.0018 | +0.0039 | no |
| 3. Ranking by HistGradientBoostingRegressor | 0.5982 ± 0.0019 | −0.0034 | no |
| 3. Average of classifier + regressor ranks | 0.6026 ± 0.0010 | +0.0009 | no |
| 4. Quotas per user_id instead of (user_id, sport) | 0.6006 ± 0.0006 | −0.0010 | no |
| 5. Outlier altitude_mean_m (15 values) → NaN | 0.6016 ± 0.0015 | −0.0001 | no |
| Off-protocol: 1a + 2 together | 0.6067 ± 0.0005 | +0.0050 | **adopted** by team decision (gain on all 3 seeds; chosen after the fact, so to be confirmed on the public score) |

All the features that use the labels (kernel, correlations) are computed in nested OOF, like the temporal neighbours.
Takeaway: the new features go in the right direction (+0.002 to +0.004) but remain below the threshold.
Quotas per (user, sport) are better than per user, which confirms that the tertiles are cut per (user, sport).

## Confusion matrix (Gate 2)

Rows = true class, columns = predicted class, on the out-of-fold predictions of the best model after post-processing.

In [10]:
best = max(results, key=lambda k: results[k][0]["full"])
cm = confusion_matrix(y_train, results[best][1]["full"])
print("meilleur modèle :", best)
print(pd.DataFrame(cm, index=["vrai " + c for c in CLASSES], columns=["prédit " + c for c in CLASSES]))

meilleur modèle : hist_gb
               prédit easy  prédit moderate  prédit hard
vrai easy             4955             1958          616
vrai moderate         1833             3839         1931
vrai hard              618             1822         5156


## Step 6: final training and submission

Same scheme as a CV fold, but with `fit` = the whole train set and `apply` = the test set:
OOF neighbour features on the train set, test neighbour features from all the train labels,
then duplicates and quotas.

In [11]:
X_fit = make_X(TRAIN_IDX, oof_label_features(TRAIN_IDX))
X_test = make_X(TEST_IDX, label_features(TRAIN_IDX, TEST_IDX))
model = bagged_hgb().fit(X_fit, y_train)      # v4 : 5 HGB moyennés (le meilleur modèle de la CV est hist_gb)
proba_test = model.predict_proba(X_test)
fixed_test = near_duplicate_labels(TRAIN_IDX, TEST_IDX, duplicate_labels(TRAIN_IDX, TEST_IDX))
pred_test = balance_ord(TRAIN_IDX, TEST_IDX, proba_test, fixed_test)

submission = pd.DataFrame({"workout_id": test["workout_id"], "intensity": np.array(CLASSES)[pred_test]})
submission.to_csv("submission.csv", index=False)
print(submission["intensity"].value_counts(normalize=True).round(3))
print("doublons copiés depuis le train :", (fixed_test >= 0).sum())

intensity
hard        0.338
easy        0.332
moderate    0.330
Name: proportion, dtype: float64
doublons copiés depuis le train : 398
